# Detección de Acordes Polifónicos de Guitarra Acústica de Cuerdas de 

El objetivo de este proyecto es desarrollar un sistema basado en Inteligencia Artificial capaz de "escuchar" una pista de audio de guitarra acústica y transcribir automáticamente los acordes que están sonando.

### Metodología
Este proyecto aborda el problema de la transcripción musical automática estructurándolo como un flujo de clasificación supervisada de imágenes:
1. **Preprocesamiento de Audio (Librosa):** Transformamos las ondas de audio unidimensionales (`.wav`) al dominio de frecuencia-tiempo utilizando la **Transformada Constant-Q (CQT)**. Esto genera espectrogramas en escala logarítmica (dB) diseñados para representar fielmente las notas y armónicos musicales.

2. **Dataset (GuitarSet):** Utilizamos el dataset de código abierto *GuitarSet*, del cual segmentamos ventanas de 0.5 segundos y sus correspondientes anotaciones extraídas de archivos `.jams`.

3. **Clasificación (PyTorch):** Reducimos la taxonomía a **25 clases** fundamentales (12 Mayores, 12 Menores y Sin Acorde) y entrenamos una **Red Neuronal Convolucional 2D (CNN)** para que aprenda los patrones visuales y armónicos que definen cada acorde.

In [ ]:
%pip install torch librosa pandas scikit-learn seaborn matplotlib ipywidgets

In [2]:
# 1. Utilidades de sistema y Python estándar
import os
import json
import glob
import time

# 2. Computación numérica y matemáticas
import numpy as np

# 3. Procesamiento de audio y representación musical
import librosa
import librosa.display

# 4. PyTorch (Deep Learning y DataLoaders)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 5. Interfaces de usuario y visualización en Jupyter
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
import matplotlib.pyplot as plt

# Configuraciones adicionales
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)


## Análisis Exploratorio del Dataset

#### Directorio

In [3]:
# Constants and path utilities
PATH = "GuitarSet/3371780"

def get_audio_folders(base_path):
    """Returns a sorted list of audio directories within the base path."""
    return sorted([
        f for f in os.listdir(base_path) 
        if os.path.isdir(os.path.join(base_path, f)) and f.startswith("audio")
    ])

def get_audio_files(folder_path):
    """Returns a sorted list of .wav files in the specified directory."""
    return sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])


#### Explorador y visualizador del dataset

In [4]:
# Audio processing and visualization state
class AudioVisualizer:
    """Handles audio loading, CQT computation, and spectrogram rendering."""
    
    def __init__(self):
        self.cqt_db = None
        self.sr = 44100
        self.hop_length = 512
        self.fmin = librosa.note_to_hz('C2')
        self.duration = 0.0

    def load_and_compute(self, file_path):
        """Loads audio and computes CQT, storing it in memory to optimize panning/zooming."""
        y, self.sr = librosa.load(file_path, sr=44100)
        self.duration = librosa.get_duration(y=y, sr=self.sr)
        
        n_bins = 84
        cqt_matrix = librosa.cqt(
            y, sr=self.sr, fmin=self.fmin, n_bins=n_bins, 
            bins_per_octave=12, hop_length=self.hop_length
        )
        self.cqt_db = librosa.amplitude_to_db(np.abs(cqt_matrix), ref=np.max)

    def plot_spectrogram(self, start_time, window_size):
        """Displays the computed spectrogram sliced by the current time window."""
        if self.cqt_db is None:
            return
            
        fig, ax = plt.subplots(figsize=(14, 6))
        img = librosa.display.specshow(
            self.cqt_db, sr=self.sr, x_axis='time', y_axis='cqt_note', 
            fmin=self.fmin, cmap='coolwarm', ax=ax, hop_length=self.hop_length
        )
        
        # Apply time axis boundaries
        ax.set_xlim(start_time, start_time + window_size)
        
        ax.set_title('Constant-Q Transform (CQT) Spectrogram')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Pitch')
        fig.colorbar(img, ax=ax, format="%+2.0f dB")
        
        plt.tight_layout()
        plt.show()

# Global instance to persist state across widget interactions
visualizer = AudioVisualizer()


In [5]:
# Interactive explorer interface

# Internal state for zoom level
zoom_state = {'window_size': 10.0}

audio_folders = get_audio_folders(PATH)

# File selection controls
folder_dropdown = widgets.Dropdown(
    options=audio_folders, description='Folder:', 
    layout=widgets.Layout(width='300px')
)

file_dropdown = widgets.Dropdown(
    options=[], description='Track:', 
    layout=widgets.Layout(width='400px')
)

# Navigation controls
zoom_in_btn = widgets.Button(
    description='Zoom In (+)', icon='search-plus', 
    layout=widgets.Layout(width='120px')
)

zoom_out_btn = widgets.Button(
    description='Zoom Out (-)', icon='search-minus', 
    layout=widgets.Layout(width='120px')
)

pan_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=10.0, step=0.1,
    description='Timeline:',
    continuous_update=False, # Wait for drag release before rendering
    layout=widgets.Layout(width='600px')
)

# Display areas
output_audio = widgets.Output()
output_plot = widgets.Output()

def update_slider_bounds():
    """Adjusts the panning slider constraints based on duration and zoom window."""
    if visualizer.duration > 0:
        max_start = max(0.0, visualizer.duration - zoom_state['window_size'])
        pan_slider.max = max_start
        if pan_slider.value > max_start:
            pan_slider.value = max_start

def on_folder_change(change):
    """Updates the track list when the source folder changes."""
    selected_folder = change['new']
    if selected_folder:
        folder_path = os.path.join(PATH, selected_folder)
        files = get_audio_files(folder_path)
        file_dropdown.options = files
        if files:
            file_dropdown.value = files[0]

def on_file_change(change):
    """Loads new audio, resets zoom state, and triggers initial render."""
    selected_folder = folder_dropdown.value
    selected_file = file_dropdown.value
    
    if not selected_folder or not selected_file:
        return
        
    file_path = os.path.join(PATH, selected_folder, selected_file)
    
    with output_audio:
        clear_output(wait=True)
        display(Audio(filename=file_path))
        
    visualizer.load_and_compute(file_path)
    
    # Reset zoom and timeline defaults
    zoom_state['window_size'] = min(10.0, visualizer.duration)
    
    pan_slider.unobserve(on_pan_change, names='value')
    pan_slider.value = 0.0
    update_slider_bounds()
    pan_slider.observe(on_pan_change, names='value')
    
    render_plot()

def on_zoom_in(b):
    """Halves the visible window size and re-renders."""
    if zoom_state['window_size'] > 0.5:
        zoom_state['window_size'] /= 2.0
        update_slider_bounds()
        render_plot()

def on_zoom_out(b):
    """Doubles the visible window size and re-renders."""
    if zoom_state['window_size'] < visualizer.duration:
        zoom_state['window_size'] = min(visualizer.duration, zoom_state['window_size'] * 2.0)
        update_slider_bounds()
        render_plot()

def on_pan_change(change):
    """Re-renders the plot sliced at the new timeline position."""
    render_plot()

def render_plot():
    """Renders the spectrogram slice within the designated output widget."""
    with output_plot:
        clear_output(wait=True)
        visualizer.plot_spectrogram(
            start_time=pan_slider.value, 
            window_size=zoom_state['window_size']
        )

# Bind widget events
folder_dropdown.observe(on_folder_change, names='value')
file_dropdown.observe(on_file_change, names='value')
zoom_in_btn.on_click(on_zoom_in)
zoom_out_btn.on_click(on_zoom_out)
pan_slider.observe(on_pan_change, names='value')

# Trigger initial startup sequence
on_folder_change({'new': folder_dropdown.value})

# Layout assembly
controls_box = widgets.HBox([folder_dropdown, file_dropdown])
zoom_box = widgets.HBox([zoom_out_btn, zoom_in_btn, pan_slider])

display(widgets.VBox([
    controls_box, 
    output_audio,
    zoom_box,
    output_plot
]))
